In [1]:
import numpy as np
from numpy import inf

In [2]:
#given values for the problems

d = np.array([[0,10,12,11,14]
          ,[10,0,13,15,8]
          ,[12,13,0,9,14]
          ,[11,15,9,0,16]
          ,[14,8,14,16,0]])


iteration = 100
n_ants = 5
n_citys = 5

In [3]:
# Inisialisasi parameter ACO

m = n_ants
n = n_citys
e = 0.1         # Menurunkan evaporation rate agar informasi pheromone bertahan lebih lama
alpha = 1       # Pheromone factor (bobot jejak pheromone)
beta = 2        # Visibility factor (bobot kedekatan jarak)

In [4]:
#calculating the visibility of the next city visibility(i,j)=1/d(i,j)

visibility = 1/d
visibility[visibility == inf ] = 0

/tmp/ipykernel_3838/3198957494.py:3: RuntimeWarning: divide by zero encountered in divide
  visibility = 1/d


In [5]:
#intializing pheromne present at the paths to the cities

pheromne = .1*np.ones((m,n))

In [6]:
# Inisialisasi rute semut dengan ukuran (n_ants, n_citys + 1)
rute = np.zeros((m, n + 1), dtype=int)
rute[:, 0] = 0

for ite in range(iteration):
    for i in range(m):
        temp_visibility = np.array(visibility)  # Duplikat visibilitas untuk melacak kota yang dikunjungi

        for j in range(n - 1):
            cur_loc = rute[i, j]
            temp_visibility[:, cur_loc] = 0  # Kota saat ini tidak boleh dikunjungi lagi

            p_feature = np.power(pheromne[cur_loc, :], alpha)
            v_feature = np.power(temp_visibility[cur_loc, :], beta)

            combine_feature = p_feature * v_feature
            total = np.sum(combine_feature)

            if total == 0:
                unvisited = np.where(temp_visibility[cur_loc, :] > 0)[0]
                city = np.random.choice(unvisited) if len(unvisited) > 0 else cur_loc
            else:
                probs = combine_feature / total
                cum_prob = np.cumsum(probs)
                r = np.random.random_sample()
                city = np.nonzero(cum_prob > r)[0][0]

            rute[i, j + 1] = city

        # Menentukan kota terakhir yang belum dikunjungi
        left = list(set(range(n)) - set(rute[i, :-2]))[0]
        rute[i, -2] = left

        # Kembali ke kota awal (indeks 0)
        rute[i, -1] = rute[i, 0]

    dist_cost = np.zeros((m, 1))

    # Menghitung total jarak perjalanan langsung menggunakan matriks rute
    for i in range(m):
        s = 0
        for j in range(n):
            s += d[rute[i, j], rute[i, j + 1]]
        dist_cost[i] = s

    dist_min_loc = np.argmin(dist_cost)
    dist_min_cost = dist_cost[dist_min_loc][0]
    best_route = rute[dist_min_loc, :]

    # Evaporasi pheromone
    pheromne = (1 - e) * pheromne

    # Pembaruan pheromone
    for i in range(m):
        dt = 1.0 / dist_cost[i].item()
        for j in range(n):
            from_city = rute[i, j]
            to_city = rute[i, j + 1]
            pheromne[from_city, to_city] += dt

print('Rute seluruh semut di akhir iterasi :')
print(rute + 1)
print()
print('Rute terbaik (1-based) :', best_route + 1)
print('Total biaya rute terbaik  :', int(dist_min_cost))

Rute seluruh semut di akhir iterasi :
[[1 2 5 3 4 1]
 [1 2 5 3 4 1]
 [1 2 5 3 4 1]
 [1 2 5 3 4 1]
 [1 2 5 3 4 1]]

Rute terbaik (1-based) : [1 2 5 3 4 1]
Total biaya rute terbaik  : 52
